#EXTRACTING TABLES

In [0]:
%run ./01_loading_AAPL_sec_file


**FUNTION TO LOOK FOR KEYWORDS IN THE TEXT**
def searcher(keyword):
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        if keyword in text:
            print(f"Found keyword on page {i}")
%md
**FUNTION TO EXTRACT THE TABLE FROM A LIST WITH PAGE NUMBERS**
def extract_tables(list):

    pages_text = []

    for p in list:
        text = pdf.pages[p].extract_text()
        pages_text.append(text)

    rows = []

    for text in pages_text:
        for line in text.split("\n"):
            # a very rough filter: lines that contain numbers in columns
            if re.search(r"\d", line):
                rows.append(line)
    
    df = pd.DataFrame([re.split(r"\s{2,}", r.strip()) for r in rows])
    print(df.head(10))
    return df

%md
**1. EXTRACTING BALANCE SHEET**
searcher("BALANCE SHEETS")
pages = [33]
balance_sheet = extract_tables(pages)
%md
**2. EXTRACTING CASH FLOW**
searcher("CASH FLOWS")
pages = [35]

cash_flow = extract_tables(pages)
%md
**3. EXTRACTING INCOME STATEMENT**
searcher("CONSOLIDATED STATEMENTS OF OPERATIONS")
pages = [31]

income_statement = extract_tables(pages)

## PACKAGES AND LIBRARIES

In [0]:
import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

## FUNTIONS

### SEARCH FOR KEYWORDS IN THE PDF AND RETURNING PAGES

In [0]:
def searcher(keyword):
    pages = []

    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        if keyword in text:
            pages.append(i)
            print(f"Found keyword on page {i}")
    
    return pages

### EXTRACT TABLES FROM A LIST WITH PAGE NUMBERS (PANDAS)

In [0]:
def extract_tables(list): #For Pandas Dataframes

    pages_text = []

    for p in list:
        text = pdf.pages[p].extract_text()
        pages_text.append(text)

    rows = []

    for text in pages_text:
        for line in text.split("\n"):
            # a very rough filter: lines that contain numbers in columns
            if re.search(r"\d", line):
                rows.append(line)
    
    df = pd.DataFrame([re.split(r"\s{2,}", r.strip()) for r in rows])
    print(df.head(10))
    return df


### EXTRACT TABLES FROM A LIST WITH PAGE NUMBERS (PY.SPARK)

In [0]:
def extract_tables_pyspark(pdf, page_list, spark):
    # 1. Extract text from selected pages
    pages_text = []
    for p in page_list:
        text = pdf.pages[p].extract_text()
        pages_text.append(text)

    # 2. Split text into lines and filter lines containing digits
    lines = []
    for text in pages_text:
        for line in text.split("\n"):
            if re.search(r"\d", line):   # keeps only rows containing numbers
                lines.append(line)

    # 3. Convert Python list → Spark DataFrame
    df = spark.createDataFrame([(line,) for line in lines], ["raw"])

    # 4. Split on 2 or more spaces using Spark regexp
    df = df.withColumn("cols", F.split(F.regexp_replace("raw", r"\s{2,}", "||"), r"\|\|"))

    # 5. Drop original raw column
    df = df.drop("raw")

    return df

## SEARCHING AND EXTRACTING TABLES

###1. EXTRACTING BALANCE SHEET

In [0]:
pages = searcher("BALANCE SHEETS")

In [0]:
balance_sheet = extract_tables(pages)

###2. EXTRACTING CASH FLOW

In [0]:
pages = searcher("CASH FLOWS")

In [0]:
cash_flow = extract_tables(pages)

###3. EXTRACTING INCOME STATEMENT

In [0]:
pages = searcher("CONSOLIDATED STATEMENTS OF OPERATIONS")

In [0]:
income_statement = extract_tables(pages)

###4. DISPLAYING THE TABLES

In [0]:
income_statement.display()
balance_sheet.display()
cash_flow.display()